# ASR Shootout — Colab GPU notebook

Runs the two **open-source / GPU-only** models that don't fit on your Mac:

1. `openai/whisper-large-v3`  → saved as `whisper_large_v3_local`
2. `ai4bharat/indic-conformer-600m-multilingual`  → saved as `indic_conformer`

**Runtime → Change runtime type → T4 GPU** before running.

## How to use
1. Upload your local project as a zip (or git push and clone here).
2. Make sure `data/audio/` has your 20 wav files and `data/ground_truth.json` is present.
3. Run all cells top to bottom.
4. Download the `results/transcripts/` folder back into your local project.
5. Locally run `python -m src.analyze`.

## 1. Upload project + install deps

In [ ]:
# Option A: upload a zip of your local asr-shootout/ folder
from google.colab import files
import os, zipfile, shutil
uploaded = files.upload()  # pick asr-shootout.zip
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as zf:
            zf.extractall('/content/')
        print('extracted', name)
%cd /content/asr-shootout

In [ ]:
!pip -q install -r requirements_colab.txt

In [ ]:
import torch, sys
print('Python:', sys.version.split()[0])
print('Torch :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Run Whisper-large-v3 on all 20 clips

In [ ]:
import json, time, traceback
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import pipeline as hf_pipeline

ROOT = Path('/content/asr-shootout')
AUDIO = ROOT / 'data/audio'
GT = json.loads((ROOT / 'data/ground_truth.json').read_text())
OUT = ROOT / 'results/transcripts/whisper_large_v3_local'
OUT.mkdir(parents=True, exist_ok=True)

wavs = {p.stem.lower(): p for p in AUDIO.iterdir() if p.suffix.lower() in {'.wav', '.m4a', '.mp3'}}
print(f'Found {len(wavs)} audio files')

pipe = hf_pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-large-v3',
    torch_dtype=torch.float16,
    device='cuda',
)

def resolve(clip):
    fn = clip['filename'].lower()
    stem = Path(fn).stem
    if stem in wavs:
        return wavs[stem]
    prefix = clip['id'] + '_'
    for k, v in wavs.items():
        if k.startswith(prefix):
            return v
    return None

for clip in tqdm(GT, desc='whisper-large-v3'):
    audio = resolve(clip)
    out_path = OUT / f"{clip['id']}.json"
    if out_path.exists():
        continue
    if audio is None:
        out_path.write_text(json.dumps({
            'model': 'whisper_large_v3_local', 'clip_id': clip['id'],
            'filename': clip['filename'], 'text': '', 'latency_s': 0.0,
            'error': 'audio_not_found', 'raw': None,
        }, ensure_ascii=False, indent=2))
        continue
    t0 = time.perf_counter()
    try:
        r = pipe(str(audio), generate_kwargs={'language': 'hi', 'task': 'transcribe'},
                 return_timestamps=False)
        text = r['text'].strip() if isinstance(r, dict) else str(r).strip()
        err = None
    except Exception as e:
        text, err = '', f'{type(e).__name__}: {e}\n{traceback.format_exc()}'
    out_path.write_text(json.dumps({
        'model': 'whisper_large_v3_local', 'clip_id': clip['id'],
        'filename': audio.name, 'text': text,
        'latency_s': time.perf_counter() - t0,
        'error': err, 'raw': {'text': text},
    }, ensure_ascii=False, indent=2))
print('Whisper done →', OUT)

## 3. Run AI4Bharat IndicConformer 600M

If this cell errors (NeMo install issues), run the **fallback** cell instead — vasista22/whisper-hindi-large-v2 — which preserves the "India-tuned open-source" slot.

In [ ]:
import json, time, traceback
from pathlib import Path
from tqdm.auto import tqdm
import torch, torchaudio
from transformers import AutoModel

ROOT = Path('/content/asr-shootout')
AUDIO = ROOT / 'data/audio'
GT = json.loads((ROOT / 'data/ground_truth.json').read_text())
OUT = ROOT / 'results/transcripts/indic_conformer'
OUT.mkdir(parents=True, exist_ok=True)

model = AutoModel.from_pretrained(
    'ai4bharat/indic-conformer-600m-multilingual', trust_remote_code=True
).cuda()
model.eval()

wavs = {p.stem.lower(): p for p in AUDIO.iterdir() if p.suffix.lower() in {'.wav', '.m4a', '.mp3'}}

def resolve(clip):
    stem = Path(clip['filename']).stem.lower()
    if stem in wavs: return wavs[stem]
    prefix = clip['id'] + '_'
    for k,v in wavs.items():
        if k.startswith(prefix): return v
    return None

for clip in tqdm(GT, desc='indic-conformer'):
    audio = resolve(clip)
    out_path = OUT / f"{clip['id']}.json"
    if out_path.exists(): continue
    if audio is None:
        out_path.write_text(json.dumps({'model':'indic_conformer','clip_id':clip['id'],
            'filename':clip['filename'],'text':'','latency_s':0.0,'error':'audio_not_found','raw':None}, ensure_ascii=False, indent=2)); continue
    try:
        wav, sr = torchaudio.load(str(audio))
        if sr != 16000: wav = torchaudio.functional.resample(wav, sr, 16000)
        if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
        wav = wav.cuda()
        t0 = time.perf_counter()
        with torch.no_grad():
            text = model(wav, 'hi', 'ctc')
        if isinstance(text, (list, tuple)): text = text[0]
        lat = time.perf_counter() - t0; err = None
    except Exception as e:
        text, lat, err = '', 0.0, f'{type(e).__name__}: {e}\n{traceback.format_exc()}'
    out_path.write_text(json.dumps({
        'model':'indic_conformer','clip_id':clip['id'],'filename':audio.name,
        'text': str(text).strip(),'latency_s': lat,'error': err,
        'raw':{'decoding':'ctc','lang':'hi'}}, ensure_ascii=False, indent=2))
print('IndicConformer done →', OUT)

## 3b. Fallback only if cell 3 failed: vasista22/whisper-hindi-large-v2
Skip this cell if IndicConformer ran cleanly.

In [ ]:
import json, time, traceback
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import pipeline as hf_pipeline

ROOT = Path('/content/asr-shootout')
GT = json.loads((ROOT / 'data/ground_truth.json').read_text())
OUT = ROOT / 'results/transcripts/indic_whisper_fallback'
OUT.mkdir(parents=True, exist_ok=True)

wavs = {p.stem.lower(): p for p in (ROOT/'data/audio').iterdir() if p.suffix.lower() in {'.wav','.m4a','.mp3'}}
pipe = hf_pipeline('automatic-speech-recognition', model='vasista22/whisper-hindi-large-v2',
                   torch_dtype=torch.float16, device='cuda')

def resolve(clip):
    stem = Path(clip['filename']).stem.lower()
    if stem in wavs: return wavs[stem]
    prefix = clip['id'] + '_'
    for k,v in wavs.items():
        if k.startswith(prefix): return v
    return None

for clip in tqdm(GT, desc='indic-whisper-fallback'):
    audio = resolve(clip)
    out_path = OUT / f"{clip['id']}.json"
    if out_path.exists(): continue
    if audio is None: continue
    t0 = time.perf_counter()
    try:
        r = pipe(str(audio), return_timestamps=False)
        text = r['text'].strip() if isinstance(r, dict) else str(r).strip(); err=None
    except Exception as e:
        text, err = '', f'{type(e).__name__}: {e}'
    out_path.write_text(json.dumps({
        'model':'indic_whisper_fallback','clip_id':clip['id'],'filename':audio.name,
        'text':text,'latency_s':time.perf_counter()-t0,'error':err,'raw':{'text':text}
    }, ensure_ascii=False, indent=2))
print('Fallback done →', OUT)

## 4. Download transcripts back to your machine

In [ ]:
import shutil
shutil.make_archive('/content/colab_transcripts', 'zip', '/content/asr-shootout/results/transcripts')
from google.colab import files as gfiles
gfiles.download('/content/colab_transcripts.zip')